# V16 · 快速分批兑现的保存账本复核

## tl;dr
三格普通Python复核251病例、462控制、原154组三对照和97未匹配机会。
原15m版完成交易251笔、净均值-16.662360573261154bp；
加一次50%分批版完成交易251笔、净均值-15.1939136994643bp。
D已知251/251，均值1.4684468737968517bp；
I已知154/251，均值2.474741942695898bp。
未知不补零，D使用相同已知配对；不是不同时间周期的替换，也不是新入场筛选。
本结论只说明保存账本核对；均值变化不等于盈利确认。
复用同一验证器，未重做原始路径或p值；Jupyter与完整schema仍未验证。

## Context & Methods
原1h大实体/吞没穿SMA40直接入场、K1极值硬止损、原72小时保持不变。
两版都按原生15分钟SMA40(HL2)真实同向→反向退出；候选只增加一次原始仓位50%兑现：
快5分钟真实翻色时，最新已完成慢15分钟颜色仍同向，且实际成交open的整仓毛收益严格>20bp。
这不是把15分钟换成5分钟，也不加新入场门。剩余仓位沿用原15分钟规则。

### Key Assumptions
预注册要求两版713条路径的最终退出时间、价格、原因、持仓时长、MFE/MAE、慢线触发
和串行占用完全一致。保存表验证器逐一检验；差异只能来自实际提前兑现与加权收益。
完成整仓的20bp成本按partial/remainder权重分摊，不把两次部分退出当成两笔整仓成本。
在该20bp门槛下旧赢家不能转亏，发生partial的旧亏只能改善；30bp压力测试不保证这一点。
已兑现partial但剩余仓位未知时，全单仍未知，不用局部利润补成完整收益。

## Data
固定summary和自足V16验证器SHA；两臂各六表及三个delta共十五份CSV逐一验证output_hashes。
仅调用verify_tables(tables, summary)纯表函数，不调用整仓CLI/main，不加载raw或其他实验结果。
从仓库运行或设置NOTEBOOK_REPOSITORY_ROOT；代码格只使用Python标准库。

In [1]:
import csv, gzip, hashlib, importlib.util, io, json
from pathlib import Path
RESULTS_RELATIVE='experiments/active/exp-btcusdtp-1h-dual-partial-preholdout-20260906-v16/results'
EVIDENCE_FILES=('baseline/case_trades.csv.gz', 'baseline/control_trades.csv.gz', 'baseline/case_episodes.csv.gz', 'baseline/control_episodes.csv.gz', 'baseline/matched.csv', 'baseline/single_pending.csv.gz', 'candidate/case_trades.csv.gz', 'candidate/control_trades.csv.gz', 'candidate/case_episodes.csv.gz', 'candidate/control_episodes.csv.gz', 'candidate/matched.csv', 'candidate/single_pending.csv.gz', 'case_delta.csv', 'excess_delta.csv', 'serial_delta.csv')
TABLE_FILES={'case_trades': 'case_trades.csv.gz', 'control_trades': 'control_trades.csv.gz', 'case_episodes': 'case_episodes.csv.gz', 'control_episodes': 'control_episodes.csv.gz', 'matched': 'matched.csv', 'single_pending': 'single_pending.csv.gz'}
DELTA_NAMES=('case_delta', 'excess_delta', 'serial_delta')
VERIFIER_FILES=('scripts/verify_hourly_impulse_dual_partial_v16.py',)
SUMMARY_SHA256='51ae3a00ff50b2d0a485d2ae8f37ecd4d97a0fc2bd68f6598a064b97a0876742'
VERIFIER_HASHES={'scripts/verify_hourly_impulse_dual_partial_v16.py': 'ca9566e39bff8aa5bae836a3cc979ab0052548e4e039c13f349f8271c4ab30a0'}
def require(ok,message):
    if not ok:raise ValueError(message)
def digest(data):return hashlib.sha256(data).hexdigest()
hint=globals().get("NOTEBOOK_REPOSITORY_ROOT")
roots=[Path(hint)] if hint is not None else [Path.cwd(),*Path.cwd().parents]
root=next((p.resolve() for p in roots if (p/RESULTS_RELATIVE/"summary.json").is_file()),None)
require(root is not None,"Run from repository or set NOTEBOOK_REPOSITORY_ROOT")
directory=(root/RESULTS_RELATIVE).resolve()
require(directory.is_relative_to(root),"Evidence escaped repository")
def evidence_path(name):
    require(name in ("summary.json",*EVIDENCE_FILES),"Evidence not allowlisted")
    path=(directory/name).resolve()
    require(path==directory/name,"Evidence symlink changed fixed identity")
    return path
def verifier_path(name):
    require(name in VERIFIER_FILES,"Verifier not allowlisted")
    path=(root/name).resolve()
    require(path==root/name,"Verifier symlink changed identity")
    return path
print("Saved-ledger evidence only:",RESULTS_RELATIVE)

Saved-ledger evidence only: experiments/active/exp-btcusdtp-1h-dual-partial-preholdout-20260906-v16/results


### 1. 固定来源与唯一兑现开关

In [2]:
payload=evidence_path("summary.json").read_bytes()
require(digest(payload)==SUMMARY_SHA256,"Pinned summary hash mismatch")
def reject_constant(value):raise ValueError("Nonfinite JSON: "+value)
summary=json.loads(payload,parse_constant=reject_constant)
require(summary["experiment_id"]=='exp-btcusdtp-1h-dual-partial-preholdout-20260906-v16',"Wrong V16 experiment")
require(summary["status"]=="diagnostic_only_no_candidate_acceptance","Unexpected acceptance claim")
for flag in ("holdout_consumed","audit_prices_loaded","training_eligible","production_eligible","all_financial_gates_pass"):
    require(summary[flag] is False,"Unexpected safety/eligibility flag: "+flag)
require(abs(summary["known_coverage_ceiling"]-154/251)<1e-12,"Original matching support changed")
expected_policies={"baseline":{'id': '15m_native40', 'management_minutes': 15, 'ma_kind': 'SMA', 'ma_length': 40, 'exit_mode': 'transition_colour', 'confirmations': 1},"candidate":{'id': '15m_native40_dual_partial', 'management_minutes': 15, 'ma_kind': 'SMA', 'ma_length': 40, 'exit_mode': 'transition_colour', 'confirmations': 1, 'fast_partial_fraction': 0.5}}
require(set(summary["arms"])==set(expected_policies),"Wrong arms")
for arm,policy in expected_policies.items():
    require(json.dumps(summary["arms"][arm]["policy"],sort_keys=True)==json.dumps(policy,sort_keys=True),"Dual partial policies changed")
loaded={}
for name in EVIDENCE_FILES:
    data=evidence_path(name).read_bytes()
    require(digest(data)==summary["output_hashes"][name],"CSV hash mismatch: "+name)
    text=gzip.decompress(data).decode() if name.endswith(".gz") else data.decode()
    reader=csv.DictReader(io.StringIO(text))
    require(reader.fieldnames and len(reader.fieldnames)==len(set(reader.fieldnames)),"Invalid CSV headers")
    rows=list(reader)
    require(all(None not in r and all(v is not None for v in r.values()) for r in rows),"Malformed CSV")
    loaded[name]=rows
for name in VERIFIER_FILES:
    require(digest(verifier_path(name).read_bytes())==VERIFIER_HASHES[name],"Verifier dependency hash mismatch: "+name)
tables={arm:{key:loaded[arm+"/"+file] for key,file in TABLE_FILES.items()} for arm in ("baseline","candidate")}
tables.update({key:loaded[key+".csv"] for key in DELTA_NAMES})
print("Pinned summary,",len(loaded),"saved CSVs and self-contained verifier verified")

Pinned summary, 15 saved CSVs and self-contained verifier verified


## Results

### 2. 检验最终路径不变、兑现成本和三种配对差值

In [3]:
spec=importlib.util.spec_from_file_location("_v16_notebook_saved_verifier",verifier_path(VERIFIER_FILES[0]))
verifier=importlib.util.module_from_spec(spec)
spec.loader.exec_module(verifier)
validation=verifier.verify_tables(tables,summary)
require(isinstance(validation,dict) and validation.get("status","passed")=="passed","Failed validation receipt")
require(validation.get("counts")=={"cases":251,"controls":462,"matched":154,"unmatched":97},"Verifier did not retain full population")
require(set(validation.get("effects",{}))==set(DELTA_NAMES),"Verifier omitted a paired effect")
require(isinstance(validation.get("accounting"),dict),"Verifier omitted accounting")
for field,value in (("unchanged_final_paths",713),("original_cost_fraction",.002),("partial_fraction",.5)):
    require(validation["accounting"].get(field)==value,"Verifier accounting contract changed: "+field)
scope_fields=("raw_replay","inferential_p_recomputed","sma_recomputed","unlogged_edges_excluded_independently")
require(all(validation.get(field) is False for field in scope_fields),"Verifier scope overclaim or missing limitation")
require(isinstance(validation.get("limitation"),str) and validation["limitation"],"Verifier omitted scope limitation")
scope={field:validation[field] for field in (*scope_fields,"limitation")}
verified={"counts":validation["counts"],"effects":validation["effects"],
    "accounting":validation["accounting"],"scope":scope,
    "baseline_mean_net_bp":summary["arms"]["baseline"]["metrics"]["mean_net_bp"],
    "candidate_mean_net_bp":summary["arms"]["candidate"]["metrics"]["mean_net_bp"],
    "baseline_events":summary["arms"]["baseline"]["metrics"]["events"],
    "candidate_events":summary["arms"]["candidate"]["metrics"]["events"],
    "raw_price_replay":False,"inferential_p_recomputed":False,"verifier_reused_not_independent":True}
print("Verified saved ledgers:",json.dumps(verified,ensure_ascii=False,allow_nan=False))
print("Same pinned verifier reused. No raw-price or inferential-p recomputation here.")

Verified saved ledgers: {"counts": {"cases": 251, "controls": 462, "matched": 154, "unmatched": 97}, "effects": {"case_delta": {"total_pairs": 251, "n": 251, "unknown_pairs": 0, "improved": 57, "worsened": 33, "unchanged": 161, "mean_bp": 1.4684468737968517, "sum_event_bp": 368.58016532300974}, "excess_delta": {"total_pairs": 251, "n": 154, "unknown_pairs": 97, "improved": 45, "worsened": 60, "unchanged": 49, "mean_bp": 2.474741942695898, "sum_event_bp": 381.1102591751683}, "serial_delta": {"total_pairs": 251, "n": 251, "unknown_pairs": 0, "improved": 57, "worsened": 32, "unchanged": 162, "mean_bp": 1.5981181955471488, "sum_event_bp": 401.12766708233437}}, "accounting": {"partial_fills": {"baseline/case": 0, "baseline/control": 0, "candidate/case": 90, "candidate/control": 135}, "unchanged_final_paths": 713, "original_cost_fraction": 0.002, "partial_fraction": 0.5}, "scope": {"raw_replay": false, "inferential_p_recomputed": false, "sma_recomputed": false, "unlogged_edges_excluded_indep

## Takeaways
D保留全部251机会；I保留原154组三对照支持及97个未匹配机会，不事后重配。
未知不能补零；已有partial不代表最终仓位已知。事件收益和不是复利账户收益。
改善也可能只是少亏；加权账本一致性不等于策略有正期望或真实成交保证。
同一批反复使用的2023–2024数据和61.35%匹配覆盖不能提供独立盈利确认，不自动部署。

### Execution gap
Plain Python top-down execution is not Jupyter-kernel execution. Minimum nbformat4.5 structure and code compilation are checked; full nbformat schema validation is not run. nbformat, nbclient and ipykernel are unavailable; no dependencies were installed.

原始K线、SMA颜色、首次合格事件及其路径真实性未在本notebook重建；推断p值也未重算。
完整Jupyter验证需在已有依赖的隔离环境运行
`python -m jupyter nbconvert --execute --to notebook --inplace path/to/dual_partial_audit.ipynb`。
本轮不安装依赖；三格普通Python不是Jupyter内核或完整schema验证。